# Chapter 40
## Spike Timing-Dependent Plasticity (STDP)
- Code by : [Abolfazl Ziaeemehr](https://github.com/Ziaeemehr)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ITNG/ModelingNeuralDynamics/blob/main/python/chapter40.ipynb)

## About this chapter

STDP changes synaptic weights according to the relative timing of pre- and
postsynaptic spikes. The examples below begin with the Abbott--Song
timing rule and an adapting RTM voltage trace, then progress through a
three-cell PING network (first with fixed E-to-E coupling, then with that
coupling evolving under STDP), and finish with a full PING network in
which every recurrent E-to-E synapse evolves under STDP.

Weight-update sign and size depend on spike order and time difference: a
presynaptic spike shortly before a postsynaptic one potentiates the
synapse, and the reverse order depresses it. For $\Delta
t=t_{\rm post}-t_{\rm pre}$, a representative pairwise rule is

$$
\Delta w = \begin{cases}
A_+\,e^{-\Delta t/\tau_+} & \Delta t>0\ \text{(post after pre)}\\[2pt]
-A_-\,e^{\Delta t/\tau_-} & \Delta t<0\ \text{(pre after post)}
\end{cases}
$$

The network implementation below approximates this with a smooth,
voltage-triggered version of the same idea: each E-cell carries an
adaptation-like trace, and every E-to-E weight is nudged up or down
whenever its pre- or postsynaptic cell crosses spike threshold, softly
clamped between 0 and a fixed upper bound. In three-cell PING, this
coupling changes the E-assembly lag and frequency; once STDP is active,
that timing and the weights coevolve.

The network sims below (`THREE_CELL_PING_1`-`THREE_CELL_PING_5` and
`PING_WITH_STDP`) integrate a few to a couple hundred coupled cells;
`THREE_CELL_PING_5` and `PING_WITH_STDP` run for 50000 explicit-Heun time
steps and are `@njit`-compiled (numba) for that reason. The shared
three-cell `odeint` model also compiles its repeatedly evaluated right-hand
side, while keeping SciPy's adaptive integrator and the same public API.

In [ ]:
import subprocess
import sys
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "modelingneuraldynamics"], check=True)

In [ ]:
import math
from types import SimpleNamespace

import numpy as np
from numpy import exp, tanh
import matplotlib.pyplot as plt
from scipy.integrate import odeint
from ipywidgets import interact
from numba import get_num_threads, njit, prange, set_num_threads
from numba.typed import List

## Abbott--Song STDP rule

`ABBOTT_SONG` plots the piecewise-exponential timing rule $F_0$ together
with a windowed version $F$ that tapers to zero as $|z|\to0$ (so two
simultaneous spikes produce no weight change). $z=t_{\rm post}-t_{\rm
pre}$; potentiation ($z>0$, black/red on the right) uses $K_+,\tau_+$ and
depression ($z<0$, left) uses $K_-,\tau_-$.

In [ ]:
def simulate_abbott_song(tau_plus=10., tau_minus=10., K_plus=0.1):
    '''Abbott--Song pairwise STDP timing rule F0(z) and its
    windowed/tapered version F(z) (matches ABBOTT_SONG/main.py).'''
    K_minus = 2 / 3 * K_plus
    z_pos = np.arange(1, 1000) / 1000 * tau_plus * 2
    F0_plus = K_plus * np.exp(-z_pos / tau_plus)
    F_plus = F0_plus * (1 - np.exp(-np.abs(z_pos) * 5 / tau_plus))
    z_neg = -z_pos
    F0_minus = -K_minus * np.exp(z_neg / tau_minus)
    F_minus = F0_minus * (1 - np.exp(-np.abs(z_neg) * 5 / tau_plus))
    return SimpleNamespace(z_pos=z_pos, F0_plus=F0_plus, F_plus=F_plus,
                            z_neg=z_neg, F0_minus=F0_minus, F_minus=F_minus,
                            K_plus=K_plus, K_minus=K_minus, tau_plus=tau_plus)


def plot_abbott_song(result):
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(result.z_pos, result.F0_plus, c='k', lw=3, alpha=0.8)
    ax.plot(result.z_pos, result.F_plus, lw=1, c='r')
    ax.plot(result.z_neg, result.F0_minus, c='k', lw=3, alpha=0.8)
    ax.plot(result.z_neg, result.F_minus, lw=1, c='r')
    ax.plot([0, 0], [-result.K_minus * 1.2, result.K_plus * 1.2], c='k', ls='--')
    ax.plot([-2 * result.tau_plus, 2 * result.tau_plus], [0, 0], c='k', ls='--')
    ax.set_xlabel("z [ms]")
    ax.set_title("$F_0$ (black) and $F$ (red)")
    ax.margins(x=0, y=0)
    return fig, ax

In [ ]:
interact(lambda tau_plus=10., K_plus=0.1: plot_abbott_song(simulate_abbott_song(tau_plus=tau_plus, K_plus=K_plus)),
         tau_plus=(2., 30., 1.), K_plus=(0.02, 0.3, 0.01));

## Shared cell models (used by every example below)

Every excitatory cell in this chapter is the same RTM neuron and every
inhibitory cell the same WB neuron used throughout the book (Chapter 5),
and every chemical synapse uses the two-gate release scheme from Chapter
20: a fast rise gate $q$ opens after a spike and closes with time
constant `tau_d_q`, driving a slower gate $s$ (rise `tau_r`, decay
`tau_d`) that is the actual synaptic conductance. `tau_d_q_function`
picks `tau_d_q` (by bisection) so that $s$'s peak occurs at a prescribed
`tau_peak` after the presynaptic spike.

In [ ]:
# ------------------------------------------------------------- E cell (RTM)


@njit
def m_e_inf(v):
    alpha_m = 0.32 * (v + 54) / (1 - exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


@njit
def h_e_inf(v):
    alpha_h = 0.128 * exp(-(v + 50) / 18)
    beta_h = 4. / (1 + exp(-(v + 27) / 5))
    return alpha_h / (alpha_h + beta_h)


@njit
def tau_h_e(v):
    alpha_h = 0.128 * exp(-(v + 50) / 18)
    beta_h = 4. / (1 + exp(-(v + 27) / 5))
    return 1. / (alpha_h + beta_h)


@njit
def n_e_inf(v):
    alpha_n = 0.032 * (v + 52) / (1 - exp(-(v + 52) / 5))
    beta_n = 0.5 * exp(-(v + 57) / 40)
    return alpha_n / (alpha_n + beta_n)


@njit
def tau_n_e(v):
    alpha_n = 0.032 * (v + 52) / (1 - exp(-(v + 52) / 5))
    beta_n = 0.5 * exp(-(v + 57) / 40)
    return 1. / (alpha_n + beta_n)


# ------------------------------------------------------------- I cell (WB)


@njit
def m_i_inf(v):
    alpha_m = 0.1 * (v + 35) / (1 - exp(-(v + 35) / 10))
    beta_m = 4. * exp(-(v + 60) / 18)
    return alpha_m / (alpha_m + beta_m)


@njit
def h_i_inf(v):
    alpha_h = 0.07 * exp(-(v + 58) / 20)
    beta_h = 1. / (exp(-0.1 * (v + 28)) + 1)
    return alpha_h / (alpha_h + beta_h)


@njit
def tau_h_i(v):
    alpha_h = 0.07 * exp(-(v + 58) / 20)
    beta_h = 1. / (exp(-0.1 * (v + 28)) + 1)
    return 1. / (alpha_h + beta_h) / 5


@njit
def n_i_inf(v):
    alpha_n = -0.01 * (v + 34) / (exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * exp(-(v + 44) / 80)
    return alpha_n / (alpha_n + beta_n)


@njit
def tau_n_i(v):
    alpha_n = -0.01 * (v + 34) / (exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * exp(-(v + 44) / 80)
    return 1. / (alpha_n + beta_n) / 5


# ------------------------------------------------------------------- shared


def tau_peak_function(tau_d, tau_r, tau_d_q):
    dt_ = 0.01
    dt05_ = dt_ / 2
    s, t = 0., 0.
    s_inc = exp(-t / tau_d_q) * (1 - s) / tau_r - s * tau_d
    while s_inc > 0:
        t_old, s_inc_old = t, s_inc
        s_tmp = s + dt05_ * s_inc
        s_inc_tmp = exp(-(t + dt05_) / tau_d_q) * (1 - s_tmp) / tau_r - s_tmp / tau_d
        s = s + dt_ * s_inc_tmp
        t = t + dt_
        s_inc = exp(-t / tau_d_q) * (1 - s) / tau_r - s / tau_d
    return (t_old * (-s_inc) + t * s_inc_old) / (s_inc_old - s_inc)


def tau_d_q_function(tau_d, tau_r, tau_hat):
    tau_d_q_left = 1.
    while tau_peak_function(tau_d, tau_r, tau_d_q_left) > tau_hat:
        tau_d_q_left /= 2
    tau_d_q_right = tau_r
    while tau_peak_function(tau_d, tau_r, tau_d_q_right) < tau_hat:
        tau_d_q_right *= 2
    while tau_d_q_right - tau_d_q_left > 1e-12:
        tau_d_q_mid = (tau_d_q_left + tau_d_q_right) / 2
        if tau_peak_function(tau_d, tau_r, tau_d_q_mid) <= tau_hat:
            tau_d_q_left = tau_d_q_mid
        else:
            tau_d_q_right = tau_d_q_mid
    return (tau_d_q_left + tau_d_q_right) / 2

## RTM voltage trace with an adaptation variable

`RTM_VOLTAGE_TRACE_WITH_A` integrates a single RTM cell together with an
auxiliary adaptation-like variable $a$ satisfying $\dot a = 1 -
C\,a\,(1+\tanh(v/10))$: $a$ relaxes toward $0$ whenever the cell is
depolarized (near/above spike threshold) and grows linearly otherwise.
This is the same trace used to gate STDP updates in the network examples
below -- it tracks "how long ago did this cell last spike" in a smooth,
voltage-triggered way, without needing explicit spike-time bookkeeping.

In [ ]:
def derivative_rtm_with_a(x0, t, i_ext=1.5, C=5.):
    v, n, h, a = x0
    dv = (i_ext - 100 * h * m_e_inf(v) ** 3 * (v - 50.)
          - 80 * n ** 4 * (v - (-100.)) - 0.1 * (v - (-67.)))
    dn = (n_e_inf(v) - n) / tau_n_e(v)
    dh = (h_e_inf(v) - h) / tau_h_e(v)
    da = 1 - C * a * (1 + np.tanh(0.1 * v))
    return [dv, dn, dh, da]


def simulate_rtm_voltage_trace_with_a(i_ext=1.5, C=5., t_final=100., dt=0.01):
    v0 = -70.
    x0 = [v0, n_e_inf(v0), h_e_inf(v0), 0.]
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative_rtm_with_a, x0, t, args=(i_ext, C))
    return SimpleNamespace(t=t, v=sol[:, 0], n=sol[:, 1], h=sol[:, 2], a=sol[:, 3])


def plot_rtm_voltage_trace_with_a(result):
    fig, ax = plt.subplots(2, figsize=(7, 5), sharex=True)
    ax[0].plot(result.t, result.v, lw=2, c="k")
    ax[1].plot(result.t, result.a, lw=2, c='k')
    ax[0].set_xlim(result.t.min(), result.t.max())
    ax[0].set_ylim(-100, 50)
    ax[1].set_xlabel("time [ms]")
    ax[0].set_ylabel("v [mV]")
    ax[1].set_ylabel("a [mV]")
    ax[0].set_yticks(range(-100, 100, 50))
    ax[1].set_ylim(0, 20)
    fig.tight_layout()
    return fig, ax

In [ ]:
interact(lambda i_ext=1.5, C=5.: plot_rtm_voltage_trace_with_a(simulate_rtm_voltage_trace_with_a(i_ext=i_ext, C=C)),
         i_ext=(0.5, 3.0, 0.1), C=(1., 15., 0.5));

## Three-cell PING network (shared model)

`THREE_CELL_PING_1` through `THREE_CELL_PING_4` share one network: two RTM
E-cells (drives 0.4 and 0.8) and one WB I-cell, with fixed E-to-I, I-to-E
and I-to-I coupling and a variable, non-plastic reciprocal (or one-way)
E-to-E coupling `g_ee` between the two E-cells. Integration uses
`scipy.integrate.odeint` (adaptive-step, not explicit Heun), with its
repeatedly evaluated right-hand side compiled by numba. The first call
includes compilation; later calls and sweep points reuse it.

In [ ]:
@njit
def derivative_three_cell_ping(x0, t, num_e, num_i, g_ee, g_ei, g_ie, g_ii,
                                i_ext_e, i_ext_i, v_rev_e, v_rev_i,
                                tau_r_e, tau_d_e, tau_dq_e,
                                tau_r_i, tau_d_i, tau_dq_i):
    '''Population RHS for the shared 2-E/1-I network (matches
    THREE_CELL_PING_1..4/lib.py:derivativePopulation exactly). g_ee, g_ei,
    g_ie, g_ii and every synaptic time constant are passed explicitly
    (rather than read as module globals) so this can be used with
    `odeint(..., args=(...))` regardless of which sub-example calls it.'''
    v_e = x0[:num_e]
    h_e = x0[num_e:2 * num_e]
    n_e = x0[2 * num_e:3 * num_e]
    q_e = x0[3 * num_e:4 * num_e]
    s_e = x0[4 * num_e:5 * num_e]
    n = 5 * num_e
    v_i = x0[n:n + num_i]
    h_i = x0[n + num_i:n + 2 * num_i]
    n_i = x0[n + 2 * num_i:n + 3 * num_i]
    q_i = x0[n + 3 * num_i:n + 4 * num_i]
    s_i = x0[n + 4 * num_i:]

    I_L_e = 0.1 * (v_e + 67.0)
    I_K_e = 80 * n_e ** 4 * (v_e + 100.0)
    I_Na_e = 100 * h_e * m_e_inf(v_e) ** 3 * (v_e - 50.0)
    I_syn_e = (g_ee @ s_e) * (v_rev_e - v_e) + (g_ie @ s_i) * (v_rev_i - v_e)

    dv_e = i_ext_e - I_L_e - I_K_e - I_Na_e + I_syn_e
    dh_e = (h_e_inf(v_e) - h_e) / tau_h_e(v_e)
    dn_e = (n_e_inf(v_e) - n_e) / tau_n_e(v_e)
    dq_e = 0.5 * (1 + np.tanh(0.1 * v_e)) * (1.0 - q_e) * 10.0 - q_e / tau_dq_e
    ds_e = q_e * (1.0 - s_e) / tau_r_e - s_e / tau_d_e

    I_L_i = 0.1 * (v_i + 65.0)
    I_K_i = 9.0 * n_i ** 4 * (v_i + 90.0)
    I_Na_i = 35.0 * m_i_inf(v_i) ** 3 * h_i * (v_i - 55.0)
    I_syn_i = (g_ei @ s_e) * (v_rev_e - v_i) + (g_ii @ s_i) * (v_rev_i - v_i)

    dv_i = i_ext_i - I_Na_i - I_K_i - I_L_i + I_syn_i
    dh_i = (h_i_inf(v_i) - h_i) / tau_h_i(v_i)
    dn_i = (n_i_inf(v_i) - n_i) / tau_n_i(v_i)
    dq_i = 0.5 * (1.0 + np.tanh(0.1 * v_i)) * (1.0 - q_i) * 10 - q_i / tau_dq_i
    ds_i = q_i * (1.0 - s_i) / tau_r_i - s_i / tau_d_i

    return np.hstack((dv_e, dh_e, dn_e, dq_e, ds_e, dv_i, dh_i, dn_i, dq_i, ds_i))


def spike_detection(t, v, spike_threshold, dt):
    v = np.asarray(v)
    t_spikes = []
    for i in range(1, len(v)):
        if v[i - 1] <= spike_threshold < v[i]:
            ts = ((i - 1) * dt * (v[i - 1] - spike_threshold) + i * dt * (spike_threshold - v[i])) / (v[i - 1] - v[i])
            t_spikes.append(ts)
    return np.array(t_spikes)


def simulate_three_cell_ping(g_ee, num_e=2, num_i=1, i_ext_e=(0.4, 0.8), i_ext_i=(0.,),
                              g_ei_strength=0.125, g_ie_strength=0.25, g_ii_strength=0.25,
                              v_rev_e=0., v_rev_i=-75.,
                              tau_r_e=0.5, tau_peak_e=0.5, tau_d_e=3.,
                              tau_r_i=0.5, tau_peak_i=0.5, tau_d_i=9.,
                              t_final=500., dt=0.02, spike_threshold=-20.):
    '''Integrate the shared 2-E/1-I network with a given (possibly
    asymmetric) E-to-E coupling matrix g_ee. Fixed E-to-I/I-to-E/I-to-I
    coupling matches THREE_CELL_PING_1..4/main.py.'''
    i_ext_e = np.asarray(i_ext_e, dtype=float)
    i_ext_i = np.asarray(i_ext_i, dtype=float)
    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak_e)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    g_ei = np.zeros((num_i, num_e))
    g_ie = np.zeros((num_e, num_i))
    g_ii = np.zeros((num_i, num_i))
    g_ei[0, :] = g_ei_strength
    g_ie[:, 0] = g_ie_strength
    g_ii[0, 0] = g_ii_strength

    v_e = -70.0 * np.ones(num_e)
    h_e, n_e = h_e_inf(v_e), n_e_inf(v_e)
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)

    v_i = -75.0 * np.ones(num_i)
    h_i, n_i = h_i_inf(v_i), n_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    x0 = np.hstack((v_e, h_e, n_e, q_e, s_e, v_i, h_i, n_i, q_i, s_i))
    t = np.arange(0, t_final, dt)
    sol = odeint(derivative_three_cell_ping, x0, t,
                 args=(num_e, num_i, g_ee, g_ei, g_ie, g_ii, i_ext_e, i_ext_i,
                       v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
                       tau_r_i, tau_d_i, tau_dq_i))

    t_e_spikes = [spike_detection(t, sol[:, i], spike_threshold, dt) for i in range(num_e)]
    index = 5 * num_e
    t_i_spikes = [spike_detection(t, sol[:, index + i], spike_threshold, dt) for i in range(num_i)]

    return SimpleNamespace(t=t, sol=sol, t_e_spikes=t_e_spikes, t_i_spikes=t_i_spikes,
                            num_e=num_e, num_i=num_i, t_final=t_final, g_ee=g_ee)


def reciprocal_g_ee(g, g_reverse=None):
    '''2x2 E-to-E coupling matrix with g_ee[0,1]=g and g_ee[1,0]=g
    (or g_reverse, for the one-way case used by THREE_CELL_PING_4).'''
    g_ee = np.zeros((2, 2))
    g_ee[0, 1] = g
    g_ee[1, 0] = g if g_reverse is None else g_reverse
    return g_ee

In [ ]:
def plot_three_cell_raster(results, labels=None):
    '''One raster row per result (a simulate_three_cell_ping() output),
    I-cell spikes in blue and E-cell spikes in red, with per-cell firing
    frequency printed below the figure (folds in
    THREE_CELL_PING_1..2/rastergram.py).'''
    if not isinstance(results, (list, tuple)):
        results = [results]
    fig, axes = plt.subplots(len(results), figsize=(7, 2 * len(results)), sharex=True, squeeze=False)
    for row, result in enumerate(results):
        ax = axes[row, 0]
        num_i, num_e = result.num_i, result.num_e
        for i in range(num_i):
            ts = result.t_i_spikes[i]
            if len(ts):
                ax.plot(ts, [i] * len(ts), "b.")
        for i in range(num_e):
            ts = result.t_e_spikes[i]
            if len(ts):
                ax.plot(ts, [i + num_i] * len(ts), "r.")
        ax.set_ylabel("neuron #")
        if labels is not None:
            ax.set_title(labels[row])
        for i in range(num_i):
            freq = len(result.t_i_spikes[i]) * 1000 / result.t_final
            print(f"frequency of i cell {i}: {freq:.2f} Hz" + (f" ({labels[row]})" if labels else ""))
        for i in range(num_e):
            freq = len(result.t_e_spikes[i]) * 1000 / result.t_final
            print(f"frequency of e cell {i}: {freq:.2f} Hz" + (f" ({labels[row]})" if labels else ""))
    axes[-1, 0].set_xlabel("time [ms]")
    fig.tight_layout()
    return fig, axes

## Baseline three-cell PING (no E-to-E coupling)

`THREE_CELL_PING_1` is the baseline: the two E-cells only interact
indirectly, through the shared I-cell (no direct E-to-E synapse). With
different drives (0.4 and 0.8) they fire at different, drive-dependent
rates.

In [ ]:
result_t1 = simulate_three_cell_ping(np.zeros((2, 2)))
plot_three_cell_raster(result_t1);

In [ ]:
interact(lambda g=0.0: plot_three_cell_raster(simulate_three_cell_ping(reciprocal_g_ee(g))),
         g=(0.0, 0.4, 0.02));

## Weak vs. strong reciprocal E-to-E coupling

`THREE_CELL_PING_2` compares weak ($\bar g_{EE}=0.05$) and strong
($\bar g_{EE}=0.4$) reciprocal E-to-E coupling. Strong coupling pulls the
two E-cells into a shared, faster rhythm; weak coupling leaves them closer
to their independent, drive-set rates.

In [ ]:
result_weak = simulate_three_cell_ping(reciprocal_g_ee(0.05))
result_strong = simulate_three_cell_ping(reciprocal_g_ee(0.4))
plot_three_cell_raster([result_weak, result_strong], labels=["weak (0.05)", "strong (0.4)"]);

In [ ]:
interact(lambda g_weak=0.05, g_strong=0.4: plot_three_cell_raster(
             [simulate_three_cell_ping(reciprocal_g_ee(g_weak)), simulate_three_cell_ping(reciprocal_g_ee(g_strong))],
             labels=["weak", "strong"]),
         g_weak=(0.0, 0.4, 0.01), g_strong=(0.0, 0.4, 0.01));

## E-to-E coupling sweep (shared by THREE_CELL_PING_3 and _4)

Both remaining fixed-coupling examples sweep an E-to-E strength and, for
each value, run the network to steady state (discarding the first half of
the trace) and measure two things: `Delta`, the mean lag from each spike
of E-cell 2 (drive 0.8) to the next following spike of E-cell 1 (drive
0.4), and `freq`, E-cell 2's firing frequency.

In [ ]:
def sweep_three_cell_ping_ee(g_ee_vec, reciprocal=True, num_e=2, num_i=1,
                              i_ext_e=(0.4, 0.8), i_ext_i=(0.,), t_final=500., dt=0.02,
                              spike_threshold=-20.):
    Delta_list, freq_list = [], []
    for g in g_ee_vec:
        g_ee = reciprocal_g_ee(g) if reciprocal else reciprocal_g_ee(g, g_reverse=0.)
        result = simulate_three_cell_ping(g_ee, num_e=num_e, num_i=num_i,
                                           i_ext_e=i_ext_e, i_ext_i=i_ext_i,
                                           t_final=t_final, dt=dt, spike_threshold=spike_threshold)
        ts0 = result.t_e_spikes[0]
        ts1 = result.t_e_spikes[1]
        ts0 = ts0[ts0 > t_final / 2]
        ts1 = ts1[ts1 > t_final / 2]

        Delta, length = 0.0, 0
        for tj in ts1:
            later = ts0[ts0 > tj]
            if len(later) > 0:
                Delta += (later - tj).min()
                length += 1
        Delta_list.append(Delta / length)
        freq_list.append(1000 / np.mean(np.diff(ts1)))

    return SimpleNamespace(g_ee_vec=np.asarray(g_ee_vec), Delta=np.array(Delta_list), freq=np.array(freq_list))


def plot_ee_sweep(result, freq_ylim=None):
    fig, ax = plt.subplots(ncols=2, figsize=(8, 3.5))
    ax[0].plot(result.g_ee_vec, result.Delta, lw=2, c='k')
    ax[1].plot(result.g_ee_vec, result.freq, lw=2, c='k')
    ax[0].set_ylabel(r"$\Delta$", fontsize=13)
    ax[1].set_ylabel(r"$f$", fontsize=13)
    for a in ax:
        a.set_xlabel(r"$\bar{g_{EE}}$", fontsize=13)
        a.tick_params(labelsize=13)
    if freq_ylim is not None:
        ax[1].set_ylim(*freq_ylim)
    fig.tight_layout()
    return fig, ax

## Reciprocal E-to-E strength sweep

`THREE_CELL_PING_3` sweeps a symmetric, reciprocal E-to-E strength over
$[0,0.4)$ and tracks the E1-E2 lag and E2's frequency.

In [ ]:
sweep_t3 = sweep_three_cell_ping_ee(np.arange(0, 51) / 51 * 0.4, reciprocal=True)
plot_ee_sweep(sweep_t3);

## One-way E-to-E strength sweep

`THREE_CELL_PING_4` sweeps the same range but as a one-way E2-to-E1
connection only (`g_ee[1,0]=0`), which produces qualitatively different
lag/frequency curves from the reciprocal case above.

In [ ]:
sweep_t4 = sweep_three_cell_ping_ee(np.arange(0, 51) / 51 * 0.4, reciprocal=False)
plot_ee_sweep(sweep_t4, freq_ylim=(0, 200));

## STDP building blocks (shared by THREE_CELL_PING_5 and PING_WITH_STDP)

The remaining two examples let the E-to-E coupling itself evolve under
STDP while the network spikes. Each E-cell carries an adaptation-like
trace $a$ (as in the RTM-with-$a$ example above, but relaxing toward $0$
much faster after a spike); every E-to-E weight $g_{ij}$ is nudged toward
a fixed upper bound `B` whenever cell $j$ (presynaptic) is depolarized and
$a_j$ is small (recent spike) -- and pulled back down toward $0$ whenever
cell $i$ (postsynaptic) is depolarized -- with both nudges softly clamped
to $[0,B]$. Because these networks are integrated with a fixed-step
explicit-Heun scheme for 50000 steps, the per-timestep update is
`@njit`-compiled and its independent dense STDP rows use `prange`: scalar
mirrors of the gating functions above (`math.exp`
instead of vectorized `numpy`, `math.pow(x, 3.0)` instead of `x ** 3`) plus
the STDP increment kernel `_g_ee_derivative_s`.

In [ ]:
@njit(fastmath=True)
def _m_e_inf_s(v):
    alpha_m = 0.32 * (v + 54) / (1 - math.exp(-(v + 54) / 4))
    beta_m = 0.28 * (v + 27) / (math.exp((v + 27) / 5) - 1)
    return alpha_m / (alpha_m + beta_m)


@njit(fastmath=True)
def _h_e_inf_s(v):
    alpha_h = 0.128 * math.exp(-(v + 50) / 18)
    beta_h = 4. / (1 + math.exp(-(v + 27) / 5))
    return alpha_h / (alpha_h + beta_h)


@njit(fastmath=True)
def _tau_h_e_s(v):
    alpha_h = 0.128 * math.exp(-(v + 50) / 18)
    beta_h = 4. / (1 + math.exp(-(v + 27) / 5))
    return 1. / (alpha_h + beta_h)


@njit(fastmath=True)
def _n_e_inf_s(v):
    alpha_n = 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))
    beta_n = 0.5 * math.exp(-(v + 57) / 40)
    return alpha_n / (alpha_n + beta_n)


@njit(fastmath=True)
def _tau_n_e_s(v):
    alpha_n = 0.032 * (v + 52) / (1 - math.exp(-(v + 52) / 5))
    beta_n = 0.5 * math.exp(-(v + 57) / 40)
    return 1. / (alpha_n + beta_n)


@njit(fastmath=True)
def _m_i_inf_s(v):
    alpha_m = 0.1 * (v + 35) / (1 - math.exp(-(v + 35) / 10))
    beta_m = 4. * math.exp(-(v + 60) / 18)
    return alpha_m / (alpha_m + beta_m)


@njit(fastmath=True)
def _h_i_inf_s(v):
    alpha_h = 0.07 * math.exp(-(v + 58) / 20)
    beta_h = 1. / (math.exp(-0.1 * (v + 28)) + 1)
    return alpha_h / (alpha_h + beta_h)


@njit(fastmath=True)
def _tau_h_i_s(v):
    alpha_h = 0.07 * math.exp(-(v + 58) / 20)
    beta_h = 1. / (math.exp(-0.1 * (v + 28)) + 1)
    return 1. / (alpha_h + beta_h) / 5.


@njit(fastmath=True)
def _n_i_inf_s(v):
    alpha_n = -0.01 * (v + 34) / (math.exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * math.exp(-(v + 44) / 80)
    return alpha_n / (alpha_n + beta_n)


@njit(fastmath=True)
def _tau_n_i_s(v):
    alpha_n = -0.01 * (v + 34) / (math.exp(-0.1 * (v + 34)) - 1)
    beta_n = 0.125 * math.exp(-(v + 44) / 80)
    return 1. / (alpha_n + beta_n) / 5.


@njit(fastmath=True, inline='always')
def _g_ee_derivative_row_s(i, g_ee_now, C, K_plus, K_minus, B,
                           delta_smooth, num_e, trace_plus, trace_minus,
                           tanh_ve, out):
    post = tanh_ve[i]
    for j in range(num_e):
        g = g_ee_now[i, j]

        y = g + trace_plus[i] * K_plus[i, j]
        y_min = B[i, j] if B[i, j] < y else y
        y_soft = y_min - delta_smooth[i, j] / 2 * math.log(
            1 + math.exp(-2 * abs(B[i, j] - y) / delta_smooth[i, j]))
        y_final = y_soft - g

        z = g - K_minus[i, j] * trace_minus[j]
        z_max = z if z > 0.0 else 0.0
        z_soft = z_max + delta_smooth[i, j] / 2 * math.log(
            1 + math.exp(-2 * abs(z) / delta_smooth[i, j]))
        z_final = z_soft - g

        pre = tanh_ve[j]
        out[i, j] = C * (1 + pre) * y_final + C * (1 + post) * z_final


@njit(fastmath=True)
def _g_ee_derivative_s(v_e, g_ee_now, a_arr, tau_plus, tau_minus, C,
                        K_plus, K_minus, B, delta_smooth, num_e, out):
    '''Serial STDP derivative for the two-E-cell PING model.'''
    trace_plus = np.empty(num_e)
    trace_minus = np.empty(num_e)
    tanh_ve = np.empty(num_e)
    for i in range(num_e):
        trace_plus[i] = math.exp(-a_arr[i] / tau_plus) * (1 - math.exp(-5 * a_arr[i] / tau_plus))
        trace_minus[i] = math.exp(-a_arr[i] / tau_minus) * (1 - math.exp(-5 * a_arr[i] / tau_minus))
        tanh_ve[i] = math.tanh(v_e[i] / 10)

    for i in range(num_e):
        _g_ee_derivative_row_s(i, g_ee_now, C, K_plus, K_minus, B,
                               delta_smooth, num_e, trace_plus, trace_minus,
                               tanh_ve, out)


@njit(fastmath=True, parallel=True)
def _g_ee_derivative_parallel_s(v_e, g_ee_now, a_arr, tau_plus, tau_minus, C,
                                 K_plus, K_minus, B, delta_smooth, num_e, out):
    '''Row-parallel STDP derivative for the full E-cell population.'''
    trace_plus = np.empty(num_e)
    trace_minus = np.empty(num_e)
    tanh_ve = np.empty(num_e)
    for i in range(num_e):
        trace_plus[i] = math.exp(-a_arr[i] / tau_plus) * (1 - math.exp(-5 * a_arr[i] / tau_plus))
        trace_minus[i] = math.exp(-a_arr[i] / tau_minus) * (1 - math.exp(-5 * a_arr[i] / tau_minus))
        tanh_ve[i] = math.tanh(v_e[i] / 10)

    for i in prange(num_e):
        _g_ee_derivative_row_s(i, g_ee_now, C, K_plus, K_minus, B,
                               delta_smooth, num_e, trace_plus, trace_minus,
                               tanh_ve, out)

## Three-cell PING with STDP on the reciprocal E-to-E synapses

`THREE_CELL_PING_5` returns to the two-E/one-I network, but now the
reciprocal E-to-E weights `g_ee[0,1]` and `g_ee[1,0]` (initially equal)
evolve under the STDP rule above instead of staying fixed. Because
E-cell 2 (drive 0.8) fires faster and consistently leads E-cell 1 (drive
0.4), the synapse from 2 to 1 (`g_21`) is reliably potentiated toward its
bound `B`, while the reverse synapse (`g_12`) decays toward 0 -- and as
`g_21` strengthens, the E1-E2 lag itself shrinks.

In [ ]:
@njit(fastmath=True)
def _three_cell_ping5_loop(m_steps, dt, dt05, num_e, num_i,
                            v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
                            tau_r_i, tau_d_i, tau_dq_i, c_stdp, tau_plus, tau_minus,
                            i_ext_e, i_ext_i, g_ie, g_ei, g_ii,
                            k_plus, k_minus, b_mat, delta_smooth,
                            v_e, h_e, n_e, m_e, q_e, s_e, a, g_ee,
                            v_i, h_i, n_i, m_i, q_i, s_i):
    '''Explicit-Heun STDP stepper for the 2-E/1-I network (faithful port
    of THREE_CELL_PING_5/main.py, numba-accelerated because its 50000-step
    loop takes ~28 s in plain NumPy). Structurally identical to
    PING_WITH_STDP\'s _stdp_loop below, except: the adaptation variable a
    relaxes with rate constant 5 (not 20), spikes are detected at -40 mV
    (not -20 mV), and the two off-diagonal E-to-E weights are recorded at
    every step -- all quirks/needs of the original, non-numba script.'''
    e_times = List.empty_list(np.float64)
    e_indices = List.empty_list(np.int64)
    i_times = List.empty_list(np.float64)
    i_indices = List.empty_list(np.int64)
    g_12 = np.empty(m_steps)
    g_21 = np.empty(m_steps)

    dve = np.empty(num_e); dne = np.empty(num_e); dhe = np.empty(num_e)
    dqe = np.empty(num_e); dse = np.empty(num_e); da = np.empty(num_e)
    dvi = np.empty(num_i); dni = np.empty(num_i); dhi = np.empty(num_i)
    dqi = np.empty(num_i); dsi = np.empty(num_i)
    g_ee_inc = np.empty((num_e, num_e))

    ve_m = np.empty(num_e); ne_m = np.empty(num_e); me_m = np.empty(num_e)
    he_m = np.empty(num_e); qe_m = np.empty(num_e); se_m = np.empty(num_e)
    a_m = np.empty(num_e); gee_m = np.empty((num_e, num_e))
    vi_m = np.empty(num_i); ni_m = np.empty(num_i); mi_m = np.empty(num_i)
    hi_m = np.empty(num_i); qi_m = np.empty(num_i); si_m = np.empty(num_i)

    ve_old = np.empty(num_e)
    vi_old = np.empty(num_i)
    ee_term = np.empty(num_e); ie_term = np.empty(num_e)
    ei_term = np.empty(num_i); ii_term = np.empty(num_i)

    for step in range(m_steps):
        k = step + 1

        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_ee[j, i] * s_e[j]
            ee_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ie[j, i] * s_i[j]
            ie_term[i] = acc
        for i in range(num_i):
            acc = 0.0
            for j in range(num_e):
                acc += g_ei[j, i] * s_e[j]
            ei_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * s_i[j]
            ii_term[i] = acc

        for j in range(num_e):
            v = v_e[j]
            dve[j] = (0.1 * (-67 - v) + 80 * n_e[j] ** 4 * (-100 - v)
                      + 100 * m_e[j] ** 3 * h_e[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - n_e[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - h_e[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - q_e[j]) / 0.1 - q_e[j] / tau_dq_e
            dse[j] = q_e[j] * (1 - s_e[j]) / tau_r_e - s_e[j] / tau_d_e
            da[j] = 1 - 5 * a[j] * (1 + th)
        for j in range(num_i):
            v = v_i[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * n_i[j] ** 4 * (-90 - v)
                      + 35 * m_i[j] ** 3 * h_i[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - n_i[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - h_i[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - q_i[j]) / 0.1 - q_i[j] / tau_dq_i
            dsi[j] = q_i[j] * (1 - s_i[j]) / tau_r_i - s_i[j] / tau_d_i

        _g_ee_derivative_s(v_e, g_ee, a, tau_plus, tau_minus, c_stdp,
                            k_plus, k_minus, b_mat, delta_smooth, num_e, g_ee_inc)

        for j in range(num_e):
            ve_m[j] = v_e[j] + dt05 * dve[j]
            ne_m[j] = n_e[j] + dt05 * dne[j]
            me_m[j] = _m_e_inf_s(ve_m[j])
            he_m[j] = h_e[j] + dt05 * dhe[j]
            qe_m[j] = q_e[j] + dt05 * dqe[j]
            se_m[j] = s_e[j] + dt05 * dse[j]
            a_m[j] = a[j] + dt05 * da[j]
            for jj in range(num_e):
                gee_m[j, jj] = g_ee[j, jj] + dt05 * g_ee_inc[j, jj]
        for j in range(num_i):
            vi_m[j] = v_i[j] + dt05 * dvi[j]
            ni_m[j] = n_i[j] + dt05 * dni[j]
            mi_m[j] = _m_i_inf_s(vi_m[j])
            hi_m[j] = h_i[j] + dt05 * dhi[j]
            qi_m[j] = q_i[j] + dt05 * dqi[j]
            si_m[j] = s_i[j] + dt05 * dsi[j]

        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_ee[j, i] * se_m[j]
            ee_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ie[j, i] * si_m[j]
            ie_term[i] = acc
        for i in range(num_i):
            acc = 0.0
            for j in range(num_e):
                acc += g_ei[j, i] * se_m[j]
            ei_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * si_m[j]
            ii_term[i] = acc

        for j in range(num_e):
            v = ve_m[j]
            dve[j] = (0.1 * (-67 - v) + 80 * ne_m[j] ** 4 * (-100 - v)
                      + 100 * me_m[j] ** 3 * he_m[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - ne_m[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - he_m[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - qe_m[j]) / 0.1 - qe_m[j] / tau_dq_e
            dse[j] = qe_m[j] * (1 - se_m[j]) / tau_r_e - se_m[j] / tau_d_e
            da[j] = 1 - 5 * a_m[j] * (1 + th)
        for j in range(num_i):
            v = vi_m[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * ni_m[j] ** 4 * (-90 - v)
                      + 35 * mi_m[j] ** 3 * hi_m[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - ni_m[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - hi_m[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - qi_m[j]) / 0.1 - qi_m[j] / tau_dq_i
            dsi[j] = qi_m[j] * (1 - si_m[j]) / tau_r_i - si_m[j] / tau_d_i

        _g_ee_derivative_s(ve_m, gee_m, a, tau_plus, tau_minus, c_stdp,
                            k_plus, k_minus, b_mat, delta_smooth, num_e, g_ee_inc)

        for j in range(num_e):
            ve_old[j] = v_e[j]
        for j in range(num_i):
            vi_old[j] = v_i[j]

        for j in range(num_e):
            v_e[j] = v_e[j] + dt * dve[j]
            m_e[j] = _m_e_inf_s(v_e[j])
            h_e[j] = h_e[j] + dt * dhe[j]
            n_e[j] = n_e[j] + dt * dne[j]
            q_e[j] = q_e[j] + dt * dqe[j]
            s_e[j] = s_e[j] + dt * dse[j]
            a[j] = a[j] + dt * da[j]
            for jj in range(num_e):
                g_ee[j, jj] = g_ee[j, jj] + dt * g_ee_inc[j, jj]
        for j in range(num_i):
            v_i[j] = v_i[j] + dt * dvi[j]
            m_i[j] = _m_i_inf_s(v_i[j])
            h_i[j] = h_i[j] + dt * dhi[j]
            n_i[j] = n_i[j] + dt * dni[j]
            q_i[j] = q_i[j] + dt * dqi[j]
            s_i[j] = s_i[j] + dt * dsi[j]

        for j in range(num_e):
            if ve_old[j] < -40 and v_e[j] >= -40:
                e_indices.append(j)
                e_times.append(((v_e[j] + 40) * step * dt + (-ve_old[j] - 40) * k * dt)
                                / (v_e[j] - ve_old[j]))
        for j in range(num_i):
            if vi_old[j] < -40 and v_i[j] >= -40:
                i_indices.append(j)
                i_times.append(((v_i[j] + 40) * step * dt + (-vi_old[j] - 40) * k * dt)
                                / (v_i[j] - vi_old[j]))

        g_12[step] = g_ee[0, 1]
        g_21[step] = g_ee[1, 0]

    return e_times, e_indices, i_times, i_indices, g_12, g_21


def simulate_three_cell_ping_5(num_e=2, num_i=1, i_ext_e=(0.4, 0.8), i_ext_i=(0.0,),
                                v_rev_e=0., v_rev_i=-75.,
                                tau_r_e=0.5, tau_peak_e=0.5, tau_d_e=3.,
                                tau_r_i=0.5, tau_peak_i=0.5, tau_d_i=9.,
                                t_final=500., dt=0.01,
                                g_ee0=0.05, C=1.45, tau_plus=10., tau_minus=10.):
    '''Faithful port of THREE_CELL_PING_5/main.py.'''
    i_ext_e = np.asarray(i_ext_e, dtype=float)
    i_ext_i = np.asarray(i_ext_i, dtype=float)
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak_e)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    g_ei = np.array([[0.125], [0.125]])
    g_ie = np.array([[0.25, 0.25]])
    g_ii = np.array([[0.25]])

    g_ee = np.zeros((num_e, num_e))
    g_ee[0, 1] = g_ee0
    g_ee[1, 0] = g_ee0
    K_plus = g_ee.copy()
    K_minus = g_ee * 2 / 3
    B = 8 * g_ee
    delta_smooth = np.maximum(g_ee / 2, 1e-6)

    v_e = np.array([-70., -70.])
    m_e, h_e, n_e = m_e_inf(v_e), h_e_inf(v_e), n_e_inf(v_e)
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)
    a = 100. * np.ones(num_e)

    v_i = -75. * np.ones(num_i)
    m_i, h_i, n_i = m_i_inf(v_i), h_i_inf(v_i), n_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes, g_12, g_21 = _three_cell_ping5_loop(
        m_steps, dt, dt05, num_e, num_i,
        v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
        tau_r_i, tau_d_i, tau_dq_i, C, tau_plus, tau_minus,
        i_ext_e, i_ext_i, g_ie, g_ei, g_ii,
        K_plus, K_minus, B, delta_smooth,
        v_e, h_e, n_e, m_e, q_e, s_e, a, g_ee,
        v_i, h_i, n_i, m_i, q_i, s_i,
    )

    t_e_spikes = np.array(t_e_spikes) if len(t_e_spikes) else np.empty(0)
    i_e_spikes = np.array(i_e_spikes, dtype=int) if len(i_e_spikes) else np.empty(0, dtype=int)
    t_i_spikes = np.array(t_i_spikes) if len(t_i_spikes) else np.empty(0)
    i_i_spikes = np.array(i_i_spikes, dtype=int) if len(i_i_spikes) else np.empty(0, dtype=int)

    # lag between each spike of E-cell 2 (drive 0.8, index 1) and the next
    # following spike of E-cell 1 (drive 0.4, index 0)
    t1 = t_e_spikes[i_e_spikes == 0]
    t2 = t_e_spikes[i_e_spikes == 1]
    lags, lag_times = [], []
    for tj in t2:
        later = t1[t1 > tj]
        if len(later) > 0:
            lags.append(later.min() - tj)
            lag_times.append(tj)
    lags = np.array(lags)
    lag_times = np.array(lag_times)

    t = np.arange(1, m_steps + 1) * dt
    return SimpleNamespace(t=t, t_e_spikes=t_e_spikes, i_e_spikes=i_e_spikes,
                            t_i_spikes=t_i_spikes, i_i_spikes=i_i_spikes,
                            g_12=g_12, g_21=g_21, B=B, lags=lags, lag_times=lag_times,
                            t_final=t_final, num_e=num_e, num_i=num_i)


def plot_three_cell_ping_5(result):
    fig, axes = plt.subplots(4, 1, figsize=(8, 9))

    ax = axes[0]
    if len(result.t_i_spikes) > 0:
        ax.plot(result.t_i_spikes, result.i_i_spikes + 1, '.b', markersize=12)
    if len(result.t_e_spikes) > 0:
        ax.plot(result.t_e_spikes, result.i_e_spikes + result.num_i + 1, '.r', markersize=12)
    ax.plot([0, result.t_final], [result.num_i + 0.5, result.num_i + 0.5], '--k', linewidth=1)
    ax.set_yticks([])
    ax.axis([0, result.t_final, 0, result.num_e + result.num_i + 1])

    axes[1].plot(result.t, result.g_12, '-k', linewidth=2)
    axes[1].axis([0, result.t_final, 0, result.B[0, 1]])
    axes[1].set_ylabel(r'$\overline{g}_{syn,EE,12}$')

    axes[2].plot(result.t, result.g_21, '-k', linewidth=2)
    axes[2].axis([0, result.t_final, 0, result.B[1, 0]])
    axes[2].set_ylabel(r'$\overline{g}_{syn,EE,21}$')

    axes[3].plot(result.lag_times, result.lags, '.k', markersize=12)
    axes[3].axis([0, result.t_final, 0, 12])
    axes[3].set_xlabel('$t$ [ms]')
    axes[3].set_ylabel('lag')

    fig.tight_layout()
    return fig, axes

In [ ]:
plot_three_cell_ping_5(simulate_three_cell_ping_5());

In [ ]:
interact(lambda g_ee0=0.05: plot_three_cell_ping_5(simulate_three_cell_ping_5(g_ee0=g_ee0)),
         g_ee0=(0.01, 0.2, 0.01));

## PING network with STDP on every recurrent E-to-E synapse

`PING_WITH_STDP` scales the same idea up to a full PING network: 200 RTM
E-cells (heterogeneous drive) and 50 WB I-cells, all-to-all-ish random
E-to-I/I-to-E/I-to-I coupling, and *every* E-to-E synapse (initially
weak and uniform) evolving under the STDP rule above. Because MATLAB's
`rng('default'); rng(63806)` cannot be bit-reproduced by NumPy, this uses
its own seeded generator and is checked structurally (spike counts, and
that `vec_g` stays within `[0,B]` and develops a nontrivial spread rather
than staying at its initial value) rather than against exact reference
spike times.

In [ ]:
@njit
def _rtm_init_population(i_ext, phi_vec):
    '''Vectorized rtm_init over a population: each of len(i_ext) RTM
    neurons is integrated (Heun/midpoint) independently until its 3rd
    spike, then (v,h,n) is interpolated at phase phi_vec[i] between the
    2nd and 3rd spikes. Faithfully reproduces a quirk of the original
    script: m_tmp is computed from the pre-half-step v, not from v_tmp.'''
    num = len(i_ext)
    max_spikes = 3
    t_final_init = 2000.
    dt_ = 0.01
    dt05_ = dt_ / 2

    v = -70. * np.ones(num)
    m = m_e_inf(v)
    h = h_e_inf(v)
    n = n_e_inf(v)
    t = 0.

    num_spikes = np.zeros(num, dtype=np.int64)
    done = np.zeros(num, dtype=np.bool_)
    t_spikes = np.zeros((num, max_spikes))
    out = np.zeros((num, 3))

    g_k, g_na, g_l = 80., 100., 0.1
    v_k, v_na, v_l = -100., 50., -67.

    while np.sum(done) < num and t < t_final_init:
        v_old, h_old, n_old, t_old = v, h, n, t

        v_inc = g_k * n ** 4 * (v_k - v) + g_na * m ** 3 * h * (v_na - v) + g_l * (v_l - v) + i_ext
        h_inc = (h_e_inf(v) - h) / tau_h_e(v)
        n_inc = (n_e_inf(v) - n) / tau_n_e(v)

        v_tmp = v + dt05_ * v_inc
        m_tmp = m_e_inf(v)  # faithful port of the original script's bug (uses v, not v_tmp)
        h_tmp = h + dt05_ * h_inc
        n_tmp = n + dt05_ * n_inc

        v_inc = (g_k * n_tmp ** 4 * (v_k - v_tmp) + g_na * m_tmp ** 3 * h_tmp * (v_na - v_tmp)
                 + g_l * (v_l - v_tmp) + i_ext)
        h_inc = (h_e_inf(v_tmp) - h_tmp) / tau_h_e(v_tmp)
        n_inc = (n_e_inf(v_tmp) - n_tmp) / tau_n_e(v_tmp)

        v = v + dt_ * v_inc
        m = m_e_inf(v)
        h = h + dt_ * h_inc
        n = n + dt_ * n_inc
        t = t + dt_

        ind = np.where((v_old >= -20) & (v < -20))[0]
        for k in ind:
            num_spikes[k] += 1
            if num_spikes[k] <= max_spikes:
                t_spikes[k, num_spikes[k] - 1] = (t_old * (-20 - v[k]) + t * (v_old[k] + 20)) / (v_old[k] - v[k])

        thr = t_spikes[:, max_spikes - 1] + phi_vec * (t_spikes[:, max_spikes - 1] - t_spikes[:, max_spikes - 2])
        ind = np.where((num_spikes == max_spikes) & (t > thr) & (t_old <= thr) & (~done))[0]
        for k in ind:
            out[k, 0] = (v_old[k] * (t - thr[k]) + v[k] * (thr[k] - t_old)) / dt_
            out[k, 1] = (h_old[k] * (t - thr[k]) + h[k] * (thr[k] - t_old)) / dt_
            out[k, 2] = (n_old[k] * (t - thr[k]) + n[k] * (thr[k] - t_old)) / dt_
        done[ind] = True

    ind = np.where(~done)[0]
    out[ind, 0] = v[ind]
    out[ind, 1] = h[ind]
    out[ind, 2] = n[ind]
    return out


def rtm_init_population(i_ext, phi_vec):
    '''Initialize an RTM population from array-like drives and phases.'''
    return _rtm_init_population(np.asarray(i_ext, dtype=float),
                                np.asarray(phi_vec, dtype=float))


@njit(fastmath=True)
def _stdp_loop(m_steps, dt, dt05, num_e, num_i,
                v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
                tau_r_i, tau_d_i, tau_dq_i, c_stdp, tau_plus, tau_minus,
                i_ext_e, i_ext_i, g_ie, g_ei, g_ii,
                k_plus, k_minus, b_mat, delta_smooth,
                v_e, h_e, n_e, m_e, q_e, s_e, a, g_ee,
                v_i, h_i, n_i, m_i, q_i, s_i, lfp):
    '''Explicit-Heun PING+STDP stepper for a general (num_e, num_i)
    network (faithful port of PING_WITH_STDP/main.py; also reused by
    THREE_CELL_PING_5 above, with num_e=2 num_i=1 and a few constants
    changed -- see _three_cell_ping5_loop\'s docstring).'''
    e_times = List.empty_list(np.float64)
    e_indices = List.empty_list(np.int64)
    i_times = List.empty_list(np.float64)
    i_indices = List.empty_list(np.int64)

    dve = np.empty(num_e); dne = np.empty(num_e); dhe = np.empty(num_e)
    dqe = np.empty(num_e); dse = np.empty(num_e); da = np.empty(num_e)
    dvi = np.empty(num_i); dni = np.empty(num_i); dhi = np.empty(num_i)
    dqi = np.empty(num_i); dsi = np.empty(num_i)
    g_ee_inc = np.empty((num_e, num_e))

    ve_m = np.empty(num_e); ne_m = np.empty(num_e); me_m = np.empty(num_e)
    he_m = np.empty(num_e); qe_m = np.empty(num_e); se_m = np.empty(num_e)
    a_m = np.empty(num_e); gee_m = np.empty((num_e, num_e))
    vi_m = np.empty(num_i); ni_m = np.empty(num_i); mi_m = np.empty(num_i)
    hi_m = np.empty(num_i); qi_m = np.empty(num_i); si_m = np.empty(num_i)

    ve_old = np.empty(num_e)
    vi_old = np.empty(num_i)
    ee_term = np.empty(num_e); ie_term = np.empty(num_e)
    ei_term = np.empty(num_i); ii_term = np.empty(num_i)

    for step in range(m_steps):
        k = step + 1

        # ------------------------------------------------------- stage 1
        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_ee[j, i] * s_e[j]
            ee_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ie[j, i] * s_i[j]
            ie_term[i] = acc
        for i in range(num_i):
            acc = 0.0
            for j in range(num_e):
                acc += g_ei[j, i] * s_e[j]
            ei_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * s_i[j]
            ii_term[i] = acc

        for j in range(num_e):
            v = v_e[j]
            dve[j] = (0.1 * (-67 - v) + 80 * n_e[j] ** 4 * (-100 - v)
                      + 100 * m_e[j] ** 3 * h_e[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - n_e[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - h_e[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - q_e[j]) / 0.1 - q_e[j] / tau_dq_e
            dse[j] = q_e[j] * (1 - s_e[j]) / tau_r_e - s_e[j] / tau_d_e
            da[j] = 1 - 20 * a[j] * (1 + th)
        for j in range(num_i):
            v = v_i[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * n_i[j] ** 4 * (-90 - v)
                      + 35 * m_i[j] ** 3 * h_i[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - n_i[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - h_i[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - q_i[j]) / 0.1 - q_i[j] / tau_dq_i
            dsi[j] = q_i[j] * (1 - s_i[j]) / tau_r_i - s_i[j] / tau_d_i

        _g_ee_derivative_parallel_s(v_e, g_ee, a, tau_plus, tau_minus, c_stdp,
                            k_plus, k_minus, b_mat, delta_smooth, num_e, g_ee_inc)

        for j in range(num_e):
            ve_m[j] = v_e[j] + dt05 * dve[j]
            ne_m[j] = n_e[j] + dt05 * dne[j]
            me_m[j] = _m_e_inf_s(ve_m[j])
            he_m[j] = h_e[j] + dt05 * dhe[j]
            qe_m[j] = q_e[j] + dt05 * dqe[j]
            se_m[j] = s_e[j] + dt05 * dse[j]
            a_m[j] = a[j] + dt05 * da[j]
            for jj in range(num_e):
                gee_m[j, jj] = g_ee[j, jj] + dt05 * g_ee_inc[j, jj]
        for j in range(num_i):
            vi_m[j] = v_i[j] + dt05 * dvi[j]
            ni_m[j] = n_i[j] + dt05 * dni[j]
            mi_m[j] = _m_i_inf_s(vi_m[j])
            hi_m[j] = h_i[j] + dt05 * dhi[j]
            qi_m[j] = q_i[j] + dt05 * dqi[j]
            si_m[j] = s_i[j] + dt05 * dsi[j]

        # ------------------------------------------------------- stage 2
        for i in range(num_e):
            acc = 0.0
            for j in range(num_e):
                acc += g_ee[j, i] * se_m[j]
            ee_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ie[j, i] * si_m[j]
            ie_term[i] = acc
        for i in range(num_i):
            acc = 0.0
            for j in range(num_e):
                acc += g_ei[j, i] * se_m[j]
            ei_term[i] = acc
            acc = 0.0
            for j in range(num_i):
                acc += g_ii[j, i] * si_m[j]
            ii_term[i] = acc

        for j in range(num_e):
            v = ve_m[j]
            dve[j] = (0.1 * (-67 - v) + 80 * ne_m[j] ** 4 * (-100 - v)
                      + 100 * me_m[j] ** 3 * he_m[j] * (50 - v)
                      + ee_term[j] * (v_rev_e - v) + ie_term[j] * (v_rev_i - v) + i_ext_e[j])
            dne[j] = (_n_e_inf_s(v) - ne_m[j]) / _tau_n_e_s(v)
            dhe[j] = (_h_e_inf_s(v) - he_m[j]) / _tau_h_e_s(v)
            th = math.tanh(v / 10)
            dqe[j] = (1 + th) / 2 * (1 - qe_m[j]) / 0.1 - qe_m[j] / tau_dq_e
            dse[j] = qe_m[j] * (1 - se_m[j]) / tau_r_e - se_m[j] / tau_d_e
            da[j] = 1 - 20 * a_m[j] * (1 + th)
        for j in range(num_i):
            v = vi_m[j]
            dvi[j] = (0.1 * (-65 - v) + 9 * ni_m[j] ** 4 * (-90 - v)
                      + 35 * mi_m[j] ** 3 * hi_m[j] * (55 - v)
                      + ei_term[j] * (v_rev_e - v) + ii_term[j] * (v_rev_i - v) + i_ext_i[j])
            dni[j] = (_n_i_inf_s(v) - ni_m[j]) / _tau_n_i_s(v)
            dhi[j] = (_h_i_inf_s(v) - hi_m[j]) / _tau_h_i_s(v)
            th = math.tanh(v / 10)
            dqi[j] = (1 + th) / 2 * (1 - qi_m[j]) / 0.1 - qi_m[j] / tau_dq_i
            dsi[j] = qi_m[j] * (1 - si_m[j]) / tau_r_i - si_m[j] / tau_d_i

        # note: uses the ORIGINAL (pre-step) `a`, not `a_m` -- faithfully
        # mirroring the original script\'s reuse of the pre-step trace in
        # both the predictor and corrector calls within one step.
        _g_ee_derivative_parallel_s(ve_m, gee_m, a, tau_plus, tau_minus, c_stdp,
                            k_plus, k_minus, b_mat, delta_smooth, num_e, g_ee_inc)

        for j in range(num_e):
            ve_old[j] = v_e[j]
        for j in range(num_i):
            vi_old[j] = v_i[j]

        for j in range(num_e):
            v_e[j] = v_e[j] + dt * dve[j]
            m_e[j] = _m_e_inf_s(v_e[j])
            h_e[j] = h_e[j] + dt * dhe[j]
            n_e[j] = n_e[j] + dt * dne[j]
            q_e[j] = q_e[j] + dt * dqe[j]
            s_e[j] = s_e[j] + dt * dse[j]
            a[j] = a[j] + dt * da[j]
            for jj in range(num_e):
                g_ee[j, jj] = g_ee[j, jj] + dt * g_ee_inc[j, jj]
        for j in range(num_i):
            v_i[j] = v_i[j] + dt * dvi[j]
            m_i[j] = _m_i_inf_s(v_i[j])
            h_i[j] = h_i[j] + dt * dhi[j]
            n_i[j] = n_i[j] + dt * dni[j]
            q_i[j] = q_i[j] + dt * dqi[j]
            s_i[j] = s_i[j] + dt * dsi[j]

        lfp_sum = 0.0
        for j in range(num_e):
            if ve_old[j] > -20 and v_e[j] <= -20:
                e_indices.append(j)
                e_times.append(((-20 - v_e[j]) * step * dt + (ve_old[j] + 20) * k * dt)
                                / (ve_old[j] - v_e[j]))
            lfp_sum += v_e[j]
        for j in range(num_i):
            if vi_old[j] > -20 and v_i[j] <= -20:
                i_indices.append(j)
                i_times.append(((-20 - v_i[j]) * step * dt + (vi_old[j] + 20) * k * dt)
                                / (vi_old[j] - v_i[j]))
        lfp[k] = lfp_sum / num_e

    return e_times, e_indices, i_times, i_indices


def simulate_ping_with_stdp(num_e=200, num_i=50, seed=63806,
                             g_hat_ei=0.5, g_hat_ie=0.5, g_hat_ii=0.25,
                             p_ei=0.5, p_ie=0.5, p_ii=0.5,
                             v_rev_e=0., v_rev_i=-75.,
                             tau_r_e=0.5, tau_peak_e=0.5, tau_d_e=3.,
                             tau_r_i=0.5, tau_peak_i=0.5, tau_d_i=9.,
                             t_final=500., dt=0.01, C=1.45, tau_plus=10., tau_minus=10.):
    '''Faithful port of PING_WITH_STDP/main.py: 200 RTM E-cells (random
    heterogeneous drive) + 50 WB I-cells, random E-to-I/I-to-E/I-to-I
    coupling, every E-to-E synapse plastic under STDP.'''
    rng = np.random.default_rng(seed)
    i_ext_e = 0.2 + rng.random(num_e) * 1.8
    i_ext_i = 0.25 * np.ones(num_i)
    dt05 = dt / 2
    m_steps = round(t_final / dt)

    tau_dq_e = tau_d_q_function(tau_d_e, tau_r_e, tau_peak_e)
    tau_dq_i = tau_d_q_function(tau_d_i, tau_r_i, tau_peak_i)

    g_ee = 0.05 * np.ones((num_e, num_e)) / (num_e - 1)
    np.fill_diagonal(g_ee, 0.)
    K_plus = g_ee.copy()
    K_minus = g_ee * 2 / 3
    B = 8 * g_ee
    delta_smooth = np.maximum(g_ee / 2, 1e-6)

    u_ei = rng.random((num_e, num_i))
    u_ie = rng.random((num_i, num_e))
    u_ii = rng.random((num_i, num_i))
    g_ei = g_hat_ei * (u_ei < p_ei) / (num_e * p_ei)
    g_ie = g_hat_ie * (u_ie < p_ie) / (num_i * p_ie)
    g_ii = g_hat_ii * (u_ii < p_ii) / (num_i * p_ii)

    iv = rtm_init_population(i_ext_e, rng.random(num_e))
    v_e, h_e, n_e = iv[:, 0], iv[:, 1], iv[:, 2]
    m_e = m_e_inf(v_e)
    q_e, s_e = np.zeros(num_e), np.zeros(num_e)
    a = 100. * np.ones(num_e)

    v_i = -75. * np.ones(num_i)
    m_i, h_i, n_i = m_i_inf(v_i), h_i_inf(v_i), n_i_inf(v_i)
    q_i, s_i = np.zeros(num_i), np.zeros(num_i)

    lfp = np.zeros(m_steps + 1)
    lfp[0] = v_e.mean()

    previous_numba_threads = get_num_threads()
    set_num_threads(min(previous_numba_threads, 8))
    try:
        t_e_spikes, i_e_spikes, t_i_spikes, i_i_spikes = _stdp_loop(
            m_steps, dt, dt05, num_e, num_i,
            v_rev_e, v_rev_i, tau_r_e, tau_d_e, tau_dq_e,
            tau_r_i, tau_d_i, tau_dq_i, C, tau_plus, tau_minus,
            i_ext_e, i_ext_i, g_ie, g_ei, g_ii,
            K_plus, K_minus, B, delta_smooth,
            v_e, h_e, n_e, m_e, q_e, s_e, a, g_ee,
            v_i, h_i, n_i, m_i, q_i, s_i, lfp,
        )
    finally:
        set_num_threads(previous_numba_threads)

    t_e_spikes = np.array(t_e_spikes) if len(t_e_spikes) else np.empty(0)
    i_e_spikes = np.array(i_e_spikes, dtype=int) if len(i_e_spikes) else np.empty(0, dtype=int)
    t_i_spikes = np.array(t_i_spikes) if len(t_i_spikes) else np.empty(0)
    i_i_spikes = np.array(i_i_spikes, dtype=int) if len(i_i_spikes) else np.empty(0, dtype=int)

    # off-diagonal E-to-E synaptic strengths, as a flat vector
    mask_off_diag = ~np.eye(num_e, dtype=bool)
    vec_g = g_ee[mask_off_diag]

    return SimpleNamespace(t_e_spikes=t_e_spikes, i_e_spikes=i_e_spikes,
                            t_i_spikes=t_i_spikes, i_i_spikes=i_i_spikes,
                            vec_g=vec_g, B=B, lfp=lfp, g_ee=g_ee,
                            num_e=num_e, num_i=num_i, t_final=t_final)


def plot_ping_with_stdp_raster(result):
    fig, ax = plt.subplots(figsize=(8, 5))
    if len(result.t_i_spikes) > 0:
        ax.plot(result.t_i_spikes, result.i_i_spikes, '.b', markersize=2)
    if len(result.t_e_spikes) > 0:
        ax.plot(result.t_e_spikes, result.i_e_spikes + result.num_i, '.r', markersize=2)
    ax.plot([0, result.t_final], [result.num_i + 0.5, result.num_i + 0.5], '--k', linewidth=1)
    ax.set_yticks([result.num_i, result.num_e + result.num_i])
    ax.axis([0, result.t_final, 0, result.num_e + result.num_i + 1])
    ax.set_xlabel('$t$ [ms]')
    fig.tight_layout()
    return fig, ax


def plot_ping_with_stdp_density(result):
    '''Kernel-density estimate of the distribution of E-to-E synaptic
    strengths at the end of the simulation.'''
    vec_g = result.vec_g
    R = vec_g.max()
    sigma = 1e-4
    g_grid = -0.5 * R + np.arange(1, 301) / 300 * 2 * R - R / 300
    density = np.zeros(len(g_grid))
    for gv in vec_g:
        density += np.exp(-(g_grid - gv) ** 2 / (2 * sigma ** 2)) / np.sqrt(2 * np.pi * sigma ** 2)
    density = density / (result.num_e * (result.num_e - 1))

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(g_grid, density, '-k', linewidth=5)
    ax.set_xlabel(r'$\overline{g}_{EE}$  [mS/cm$^2$]')
    ax.set_ylabel('density [connections per mS/cm$^2$]')
    ax.axis([0, 2.5e-3, 0, density.max() * 1.2])
    fig.tight_layout()
    return fig, ax

In [ ]:
result_ping_stdp = simulate_ping_with_stdp()
plot_ping_with_stdp_raster(result_ping_stdp);

In [ ]:
plot_ping_with_stdp_density(result_ping_stdp);